In [2]:
import boto3

session = boto3.Session(profile_name="mehrnaz-dev", region_name="ap-southeast-2")


In [44]:
# aws chatbot excersise

client = session.client("bedrock-runtime")

model_id="au.anthropic.claude-haiku-4-5-20251001-v1:0"


def add_message(messages , text , role):
    new_message = {
        'role' : role ,
         'content' : [
             {
                 'text' : text
             }
         ]
    }
    messages.append(new_message)
    
def chat(messages , sys_prompt = "" , temp = 0.1 , stop = []):
    
    params = {
        "modelId": model_id,
        "messages" : messages,
        "inferenceConfig" : {
            "temperature" : temp,
            "stopSequences": stop 
         }
        
    }
     
    if sys_prompt :
        params["system"] = [{"text" : sys_prompt}]
        
    response = client.converse( **params )
          
    
    return response['output']['message']['content'][0]['text']

def chat_stream(messages , sys_prompt = "" , temp = 0.1):
    
    params = {
        "modelId": model_id,
        "messages" : messages,
        "inferenceConfig" : {
            "temperature" : temp
         }
        
    }
     
    if sys_prompt :
        params["system"] = [{"text" : sys_prompt}]
        
    response = client.converse_stream( **params )
    
    
    answer = []
    
    for event in response["stream"] : 
                if "contentBlockDelta" in event :
                    chunk = event["contentBlockDelta"]["delta"]["text"]
                    print(chunk , end="")
                    answer.append(chunk)
                    
    full_chunk = "".join(answer)
   
    
    return full_chunk



def conversation(messages , sys_prompt = "" , temp = 0.1):
    print("Assistant >> Welcome Child , Ask what you desire ! ")
    while True:
        user_input = input("You >>").strip()
        
        print("You >> " , user_input , "\n" )
        
        if user_input.lower() in ['exit' , 'quit' , 'q' , 'e' ] :
            print("Assistant >> Bye child! Come back soon")
            break
        
            
        add_message( messages , user_input , 'user')
        
        
        assistant_reponse= chat( messages , sys_prompt , temp)
        
        add_message(messages , assistant_reponse , 'assistant')
        
        print(f'Assistant >> {assistant_reponse}\n')
        
        

        

In [ ]:
# system prompt excersise

system_prompt_exc =""" You are a python engineer who writes very consice code """

messages_exc = []

add_message(messages_exc , "write a function that checks a string for duplicate characters." , 'user')

response = chat(messages_exc , system_prompt_exc)

print(response)

```python
def has_duplicates(s: str) -> bool:
    return len(s) != len(set(s))
```

**Usage:**
```python
print(has_duplicates("hello"))      # True (l appears twice)
print(has_duplicates("abc"))        # False
print(has_duplicates("aabbcc"))     # True
```

**Alternative (case-insensitive):**
```python
def has_duplicates(s: str) -> bool:
    s = s.lower()
    return len(s) != len(set(s))
```


In [ ]:
# temperature
# Temperature is a decimal value between 0 and 1 that directly influences these token selection probabilities.
# Think of it as a creativity dial:
# Low temperature (near 0): Makes the highest probability token much more likely to be selected
# High temperature (near 1): Distributes probability more evenly across all possible tokens

In [43]:
# streaming 

stream_exc = []

add_message(stream_exc, "Write a 1 sentence description of a fake database" , "user")

response = chat_stream(stream_exc)



print("\n\nTotal Message:\n" + response)


# FakeDB

FakeDB is an in-memory database system that generates realistic synthetic data on-the-fly for testing and development purposes, eliminating the need for actual production data while maintaining authentic schema structures and relationships.

Total Message:
# FakeDB

FakeDB is an in-memory database system that generates realistic synthetic data on-the-fly for testing and development purposes, eliminating the need for actual production data while maintaining authentic schema structures and relationships.


In [ ]:
# controlling model ouput


# stop sequences 
output_exc = []

add_message(output_exc ,"Count from 1 to 10" , 'user')

response = chat(output_exc , stop=["5"])

print(response)

# Stop sequences are particularly useful for:

# Controlling the length of responses
# Stopping at natural breakpoints in structured output
# Preventing Claude from continuing past specific markers or delimiters

1
2
3
4



In [47]:
# structured data

# method uses prefilling and stop sequences 

messages = []
add_message(messages, "Generate a very short event bridge rule as json" , "user")
add_message(messages, "```json" , "assistant")

text = chat(messages, stop=["```"])

print(text)



{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}



In [48]:
import json 

parse_data = json.loads(text.strip())

print(parse_data)

{'Name': 'MySimpleRule', 'EventBusName': 'default', 'EventPattern': {'source': ['aws.ec2'], 'detail-type': ['EC2 Instance State-change Notification'], 'detail': {'state': ['running']}}, 'State': 'ENABLED', 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction', 'Id': '1'}]}


In [57]:
messages = []

prompt = """
Generate three different sample AWS CLI commands, Each should be very short.
"""

prefill = "Here are the 3 commands  without any aditional comments in one block ```bash"


add_message(messages, prompt , "user")
add_message(messages, prefill , "assistant")

exc_text = chat(messages , stop=["```"])

print(exc_text)



aws s3 ls
aws ec2 describe-instances
aws lambda list-functions



In [58]:
from IPython.display import Markdown

Markdown(exc_text)


aws s3 ls
aws ec2 describe-instances
aws lambda list-functions


In [ ]:
# prompt eval workflow 

# prompt draft

# eval set ( possible inputs that we might want to put into our prompt)

# feed through claude to get actual answer

# grade them ( using grader ) -- gives number base on quality of answer

# make average of the scores (to say how well)

# iterate for number to increase

prompt = f"""
Please provide a solution to the following task:


{task}
"""


